# RigTech · Inferência em CearaMirim.tif (ciclo 1)

**Objetivo**: rodar o modelo já treinado no ciclo 1 (`modelo_dinov3_3class_c1.pt`) sobre uma imagem **de teste** (`CearaMirim.tif`) — nunca vista pelo modelo — e gerar polígonos de folha_larga / folha_estreita previstos.

**Nada é retreinado**: o backbone DINOv3 é congelado por design, e a head Transformer é carregada com os pesos salvos no ciclo 1 (`best_state` = melhor época de validação em macro-F1). Só se muda a imagem de entrada.

**Pré-requisitos no Drive**:
- `CearaMirim.tif` em qualquer lugar do MyDrive — ortomosaico RGB (bandas 1,2,3).
- `modelo_dinov3_3class_c1.pt` em qualquer lugar do MyDrive — salvo pelo `runner_colab_continuacao` ao fim do treino.
- Secret `HF_TOKEN` no Colab (chave lateral) — `dinov3-vitl16-pretrain-sat493m` é *gated*.

Runtime → GPU (T4 basta pra inferência).

## 1. Clonar repo + instalar dependências

Traz o `src/infer.py` mais recente. Se der conflito de versão, **Runtime → Restart runtime** antes da próxima célula.

In [ ]:
!git clone https://github.com/maluquintela/rigtech-weed-cycle.git /content/rigtech || (cd /content/rigtech && git pull)

!pip -q install --ignore-installed 'numpy>=1.26,<2.2' 'pandas>=2.2,<2.3'
!pip -q install rasterio geopandas shapely scikit-image tqdm transformers torch huggingface_hub

# IMPORTANTE: se aparecer erro de versao ao importar geopandas/pandas,
# faca Runtime -> Restart runtime (Ctrl+M .) antes de rodar as proximas.

## 2. Login no Hugging Face + montar Drive

In [ ]:
import os
from huggingface_hub import login

_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')

if _hf_token:
    login(token=_hf_token)
    print('Login HF OK.')
else:
    login()

from google.colab import drive
drive.mount('/content/drive')

## 3. Configuração da rodada

Só variáveis — nada muda no modelo.

In [ ]:
import os, glob

CICLO = 1  # qual modelo do ciclo usar

# procura o modelo final do ciclo em qualquer lugar do Drive
_matches = glob.glob(f'/content/drive/MyDrive/**/modelo_dinov3_3class_c{CICLO}.pt', recursive=True)
assert _matches, f'nenhum modelo_dinov3_3class_c{CICLO}.pt encontrado no Drive'
CKPT = _matches[0]

_tif_matches = glob.glob('/content/drive/MyDrive/**/CearaMirim.tif', recursive=True)
assert _tif_matches, 'CearaMirim.tif nao encontrado no Drive'
TIF = _tif_matches[0]

OUT_PREFIX = f'/content/drive/MyDrive/rigtech_ciclos/pred_cearamirim_c{CICLO}'
os.makedirs(os.path.dirname(OUT_PREFIX), exist_ok=True)

# sliding window
STRIDE = 112              # 50% de overlap (CROP=224)
WEED_THRESHOLD = 0.6      # prob min de folha_larga OU folha_estreita pra tile virar positivo
MERGE_BUFFER = 1.0        # unidades do CRS do raster (UTM: metros) -- fecha gaps entre tiles
MIN_POLY_AREA = 5.0       # descarta poligonos com area < isso (unidades^2 do CRS)
BATCH = 32

print('checkpoint:', CKPT)
print('imagem    :', TIF)
print('saidas    :', OUT_PREFIX + '.{csv,geojson,_polygons.geojson}')

## 4. Rodar inferência

Sliding window sobre o raster inteiro. Se depois quiser restringir só à plantação, gera um geojson e passa `--mask <caminho>`.

In [ ]:
%cd /content/rigtech
!python -m src.infer \
    --ckpt {CKPT} \
    --tif  {TIF} \
    --sliding --stride {STRIDE} \
    --weed-threshold {WEED_THRESHOLD} \
    --merge-buffer {MERGE_BUFFER} \
    --min-polygon-area {MIN_POLY_AREA} \
    --batch {BATCH} \
    --out-prefix {OUT_PREFIX}

## 5. Resumo dos resultados

Conta previsões por classe no CSV e polígonos agregados no geojson.

In [ ]:
import pandas as pd
import geopandas as gpd

df = pd.read_csv(OUT_PREFIX + '.csv')
print(f'total de tiles avaliados: {len(df)}')
print('\ndistribuicao por classe predita:')
print(df['pred_class'].value_counts())

print('\nprob media por classe:')
print(df[['prob_cultivo', 'prob_folha_larga', 'prob_folha_estreita']].mean())

polys = gpd.read_file(OUT_PREFIX + '_polygons.geojson')
print(f'\npoligonos agregados: {len(polys)}')
if len(polys):
    print(polys.groupby('class').agg(n=('class', 'size'), area_total=('area_crs', 'sum')))

## 5b. Repolgonizar do CSV — mais nichado

Reconstrói polígonos a partir do `.csv` já gerado (não roda o modelo de novo). Diferenças em relação ao `_polygons.geojson` original:

- **caixa `stride × stride` (112 px)** em vez de `CROP × CROP` (224 px) — polígonos ~4× menores.
- **threshold alto** (0.75 default) — só tiles em que o modelo tá bem confiante viram positivos.
- **sem merge buffer** — só une o que se toca de fato, não fecha gaps grandes.
- **área mínima maior** — descarta ruído solto.

Salva com sufixo `_tight.geojson`. Ajuste `TIGHT_THRESHOLD` / `MIN_AREA_M2` até chegar no nível de nicho que quer.

In [ ]:
import json
import pandas as pd
import rasterio
from shapely.geometry import box, mapping
from shapely.ops import unary_union
import geopandas as gpd

# ---- parametros de nicho ---------------------------------------------------
TIGHT_THRESHOLD = 0.75   # sobe pra 0.85/0.9 se quiser ainda mais rigoroso
MIN_AREA_M2 = 2.0        # descarta manchas pequenas. Em m^2 (raster em graus?
                         # a gente converte -- ver abaixo)
BOX_SIDE_PX = STRIDE     # 112 px em vez de 224 -> boxes nao se sobrepoem
MERGE_BUFFER_TIGHT = 0.0 # sem fechar gaps

TIGHT_OUT = OUT_PREFIX + '_tight.geojson'

df = pd.read_csv(OUT_PREFIX + '.csv')
print(f'tiles totais: {len(df)}')

# filtra positivos com confianca alta em folha_larga OU folha_estreita
df['best_weed_prob'] = df[['prob_folha_larga', 'prob_folha_estreita']].max(axis=1)
df['best_weed_cls']  = df[['prob_folha_larga', 'prob_folha_estreita']].idxmax(axis=1).str.replace('prob_', '')

pos = df[(df['pred_class'] != 'cultivo') & (df['best_weed_prob'] >= TIGHT_THRESHOLD)].copy()
print(f'positivos apos threshold {TIGHT_THRESHOLD}: {len(pos)}')

with rasterio.open(TIF) as src:
    px_w = abs(src.transform.a)   # tamanho de pixel (unidades do CRS)
    px_h = abs(src.transform.e)
    raster_crs = src.crs

half_w = (BOX_SIDE_PX / 2.0) * px_w
half_h = (BOX_SIDE_PX / 2.0) * px_h
print(f'box side em unidades do CRS: {2*half_w:.4f} x {2*half_h:.4f}  (crs={raster_crs})')

# se CRS eh geografico (graus), area em m^2 exige reprojetar. Fazemos isso.
is_geographic = raster_crs.is_geographic if raster_crs else False

features = []
for cls in ['folha_larga', 'folha_estreita']:
    sub = pos[pos['best_weed_cls'] == cls]
    if len(sub) == 0:
        continue
    boxes = [box(r.centroid_x - half_w, r.centroid_y - half_h,
                 r.centroid_x + half_w, r.centroid_y + half_h)
             for r in sub.itertuples()]
    if MERGE_BUFFER_TIGHT > 0:
        boxes = [b.buffer(MERGE_BUFFER_TIGHT) for b in boxes]
    merged = unary_union(boxes)
    if MERGE_BUFFER_TIGHT > 0:
        merged = merged.buffer(-MERGE_BUFFER_TIGHT)
    parts = list(merged.geoms) if merged.geom_type.startswith('Multi') else [merged]

    # area em m^2: reprojeta pra Web Mercator EPSG:3857 (aproximado mas OK pra filtro)
    gs = gpd.GeoSeries(parts, crs=raster_crs)
    gs_m = gs.to_crs('EPSG:3857') if is_geographic else gs
    areas_m2 = gs_m.area.values

    gs_out = gs.to_crs('EPSG:4326')
    for geom, a in zip(gs_out.geometry, areas_m2):
        if a < MIN_AREA_M2:
            continue
        features.append({
            'type': 'Feature',
            'geometry': mapping(geom),
            'properties': {'class': cls, 'area_m2': float(a)},
        })

print(f'poligonos finais: {len(features)}')

fc = {
    'type': 'FeatureCollection',
    'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
    'features': features,
}
with open(TIGHT_OUT, 'w') as f:
    json.dump(fc, f)
print(f'salvo: {TIGHT_OUT}')

## 6. Preview visual (opcional)

Plota os polígonos previstos sobre um thumbnail do raster pra inspeção rápida no notebook. Pra análise fina, abre o `_polygons.geojson` no QGIS por cima do `.tif`.

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
import geopandas as gpd

polys = gpd.read_file(OUT_PREFIX + '_polygons.geojson')

with rasterio.open(TIF) as src:
    scale = max(1, max(src.width, src.height) // 2000)
    out_shape = (3, src.height // scale, src.width // scale)
    thumb = src.read([1, 2, 3], out_shape=out_shape, resampling=Resampling.average)
    left, bottom, right, top = src.bounds
    raster_crs = src.crs

thumb = thumb.transpose(1, 2, 0)
if thumb.dtype != 'uint8':
    thumb = (255 * (thumb / max(thumb.max(), 1))).clip(0, 255).astype('uint8')

polys_native = polys.to_crs(raster_crs) if raster_crs is not None else polys

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(thumb, extent=(left, right, bottom, top))
colors = {'folha_larga': 'red', 'folha_estreita': 'yellow'}
for cls, color in colors.items():
    sub = polys_native[polys_native['class'] == cls]
    if len(sub):
        sub.boundary.plot(ax=ax, color=color, linewidth=1.2, label=f'{cls} ({len(sub)})')
ax.set_title(f'CearaMirim — previsoes ciclo {CICLO}')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()